# Caso de estudio 2 - Deep Learning
## Clasificación de razas de mascotas para un portal de adopción (Oxford-IIIT Pet, 35 razas)

**Equipo:** Bevc, Olivera, Tellechea · **Curso:** Inteligencia Artificial I (UCU) · **Docentes:** Juan Francisco Kurucz, Bruno Cattaneo

Este notebook corre de principio a fin, en orden, sin editar código. Lo único que hay que revisar
es la celda **1. Configuración**: ahí están las rutas a los dos paquetes (`caso2_train.zip` y `caso2_test.zip`)
y los interruptores de cada experimento.

**Qué produce** (todo en la carpeta `salidas/`):

| Archivo | Contenido |
| --- | --- |
| `split_train_val.csv` | Partición train/validación (80/20 estratificada, semilla 42) |
| `experimentos.csv` | Tabla de pruebas: preparación, modelo, métricas de validación, tiempo |
| `modelo_final.pt` + `modelo_final_meta.json` | Pesos del modelo final y su ficha (arquitectura, pesos preentrenados, clases) |
| `metricas_por_raza.csv`, `confusiones_top.csv` | Precisión, recall y F1 por raza; confusiones más frecuentes |
| `umbrales.csv`, `criterio.json` | Curva cobertura/precisión y umbral elegido para sugerir/revisar |
| `predicciones_test.csv` | Entrega final: `image_id,raza_predicha,decision` |
| `figuras/` | Todas las figuras numeradas que cita el informe |

**Datos:** Oxford-IIIT Pet (Parkhi, Vedaldi, Zisserman y Jawahar, 2012), licencia CC BY-SA 4.0, en la partición
preparada por la cátedra. Solo se usan los dos paquetes entregados; test nunca participa en entrenamiento,
selección de modelos ni ajuste del criterio de decisión.


## 1. Configuración

Único lugar con rutas y parámetros. Los datos se buscan en este orden: (1) la carpeta `data/` junto al notebook,
(2) una carpeta de Google Drive propia (`DRIVE_DATA_DIR`, se monta sola en Colab), (3) descarga de la carpeta
compartida de la cátedra. En Colab lo más confiable es la opción 2: subir los dos zips (o las carpetas descomprimidas)
a una carpeta `caso2_IA` en *Mi unidad*.


In [ ]:
from pathlib import Path

# ----- RUTAS (único lugar a revisar) ---------------------------------------
TRAIN_ZIP = Path("data/caso2_train.zip")     # paquete de entrenamiento entregado
TEST_ZIP  = Path("data/caso2_test.zip")      # paquete de test entregado
DATA_DIR  = Path("data")                      # acá se descomprimen los zips
OUT_DIR   = Path("salidas")                   # todo lo que genera el notebook
FIG_DIR   = OUT_DIR / "figuras"

# Google Drive propio (solo Colab): carpeta de "Mi unidad" donde están los zips o las carpetas descomprimidas.
# Poner None para no usar Drive.
DRIVE_DATA_DIR = "caso2_IA"      # -> /content/drive/MyDrive/caso2_IA

# IDs de los archivos en la carpeta compartida de la cátedra (último recurso: descarga pública con gdown)
DRIVE_IDS = {
    "caso2_train.zip": "1fi6v1lDBCtl8S1GWkvbqLvxUGs91g44k",
    "caso2_test.zip":  "1stq_Imc90HE5m8iUfxgu__eBwQ-oHR6c",
}

# ----- REPRODUCIBILIDAD -----------------------------------------------------
SEED = 42

# ----- PREPARACIÓN DE IMÁGENES ---------------------------------------------
IMG_SIZE    = 224       # resolución de entrada (la de preentrenamiento en ImageNet)
RESIZE_EVAL = 256       # redimensionado previo al recorte central en validación/test
VAL_FRAC    = 0.20      # proporción de train.csv reservada para validación
BATCH_SIZE  = 32
NUM_WORKERS = 2

# ----- EXPERIMENTOS (True = se corre; False = se omite y no entra en la tabla) -
RUN_E1_CNN_DESDE_CERO     = True   # camino de contraste
RUN_E2_RESNET50_FT        = True   # segunda arquitectura preentrenada
RUN_E3_CONVNEXT_FT        = True   # candidato a modelo final (con aumentos)
RUN_E4_CONVNEXT_SIN_AUG   = True   # ablación: mismo modelo sin aumentos
RUN_E5_CONVNEXT_CONGELADO = True   # solo la capa final (extracción de características)

EPOCHS = {"E1": 25, "E2": 10, "E3": 12, "E4": 12, "E5": 8}
PACIENCIA_EARLY_STOP = 4           # épocas sin mejorar F1 macro antes de cortar

# ----- CRITERIO SUGERIR / REVISAR ------------------------------------------
USE_TTA = True                     # promedio con espejado horizontal en validación y test (misma regla en ambos)
OBJETIVO_PRECISION_SUGERENCIAS = 0.95   # mínimo de aciertos entre las fotos que reciben sugerencia

# ----- MODO DE PRUEBA RÁPIDA (solo para verificar que todo corre) ----------
QUICK_TEST = False   # True: 10 % de las imágenes y 1 época por experimento. NO usar para la entrega.

RUN_GRADIO = False   # prototipo visual opcional al final

for d in (DATA_DIR, OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Configuración cargada. Semilla:", SEED, "| Imagen:", IMG_SIZE, "| QUICK_TEST:", QUICK_TEST)


## 2. Dependencias, versiones y semillas

Se registran las versiones usadas (también se escriben en `requirements.txt`) y se fijan todas las semillas
aleatorias (Python, NumPy, PyTorch y CUDA). cuDNN se pone en modo determinista; esto cuesta algo de velocidad
pero hace que dos corridas en el mismo hardware den los mismos números.


In [ ]:
import sys, subprocess, importlib, os, random, time, json, zipfile, hashlib, warnings
warnings.filterwarnings("ignore")

def _asegurar(paquete, pip_nombre=None):
    try:
        importlib.import_module(paquete)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_nombre or paquete], check=False)

for p, n in [("torch", "torch"), ("torchvision", "torchvision"), ("sklearn", "scikit-learn"),
             ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("PIL", "pillow"), ("seaborn", "seaborn")]:
    _asegurar(p, n)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
import torch, torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

sns.set_theme(style="whitegrid", font_scale=0.9)

def fijar_semillas(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # antes de inicializar CUDA
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

fijar_semillas()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

versiones = {
    "python": sys.version.split()[0], "torch": torch.__version__, "torchvision": torchvision.__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": __import__("sklearn").__version__,
    "matplotlib": matplotlib.__version__, "pillow": Image.__version__ if hasattr(Image, "__version__") else __import__("PIL").__version__,
    "seaborn": sns.__version__,
}
print("Dispositivo:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "sin GPU (el entrenamiento será lento)")
for k, v in versiones.items():
    print(f"  {k:>13}: {v}")

req = "\n".join(f"{k}=={v}" for k, v in versiones.items() if k != "python") + "\n"
(OUT_DIR / "requirements.txt").write_text(req)
(OUT_DIR / "versiones.json").write_text(json.dumps(versiones, indent=2))


## 3. Localización de los datos

Se localizan `train.csv`, `clases.csv`, `test.csv` y las imágenes sin asumir una estructura interna de carpetas,
buscando en este orden: `data/` junto al notebook, la carpeta propia de Google Drive (`DRIVE_DATA_DIR`, solo en Colab)
y, como último recurso, la descarga pública de la carpeta compartida de la cátedra. Cada zip se valida antes de
descomprimir y, si una descompresión anterior quedó incompleta, se rehace. Las imágenes se indexan por nombre de
archivo sin extensión, que es como las referencia `image_id`.


In [ ]:
import shutil

EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def _valido(p):
    return "__MACOSX" not in p.parts and not p.name.startswith("._")

def buscar_csv(raiz, nombre):
    hits = [p for p in Path(raiz).rglob(nombre) if _valido(p)]
    if not hits:
        raise FileNotFoundError(f"No se encontró {nombre} dentro de {raiz}")
    return hits[0]

def hay_csv(raiz, nombre):
    try:
        buscar_csv(raiz, nombre); return True
    except FileNotFoundError:
        return False

def indexar_imagenes(raiz):
    return {p.stem: p for p in Path(raiz).rglob("*") if p.is_file() and p.suffix.lower() in EXT_IMG and _valido(p)}

def listar(raiz, n=25):
    raiz = Path(raiz)
    if not raiz.exists():
        return f"  ({raiz} no existe)"
    items = sorted(raiz.rglob("*"))
    lineas = [f"  {p.relative_to(raiz)}{'/' if p.is_dir() else ''}" for p in items[:n]]
    if len(items) > n: lineas.append(f"  ... y {len(items) - n} más")
    return "\n".join(lineas) or "  (vacío)"

def descomprimir(zip_path, destino):
    zip_path, destino = Path(zip_path), Path(destino)
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f"{zip_path} no es un zip válido ({zip_path.stat().st_size/1e6:.1f} MB). "
                           "Si vino de una descarga, fue una descarga incompleta o una página de aviso de Drive: borrarlo y reintentar.")
    if destino.exists(): shutil.rmtree(destino)
    destino.mkdir(parents=True)
    # Los zips de la cátedra fueron armados en Windows y traen rutas con barra invertida
    # ("paquete_train_alumnos\\imagenes\\pet_x.jpg"). En Linux/Colab, extractall las toma como un nombre plano y las
    # carpetas no se crean, por lo que train.csv no aparece. Se extrae a mano normalizando los separadores.
    n = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            partes = [q for q in info.filename.replace("\\", "/").split("/") if q and q != ".."]
            if not partes or info.is_dir() or info.filename.endswith(("/", "\\")) or partes[0] == "__MACOSX":
                continue
            objetivo = destino.joinpath(*partes)
            objetivo.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(objetivo, "wb") as dst:
                shutil.copyfileobj(src, dst)
            n += 1
    print(f"{zip_path.name} -> {destino} ({n} archivos)")

def descargar_publico(nombre, destino_zip):
    print(f"Descargando {nombre} desde la carpeta compartida de la cátedra...")
    _asegurar("gdown"); import gdown
    destino_zip.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=DRIVE_IDS[nombre], output=str(destino_zip), quiet=False)
    if not destino_zip.exists() or not zipfile.is_zipfile(destino_zip):
        if destino_zip.exists(): destino_zip.unlink()
        raise RuntimeError(f"La descarga pública de {nombre} falló (Drive suele limitar descargas de archivos grandes). "
                           f"Alternativa: subir {nombre} a la carpeta '{DRIVE_DATA_DIR}' de tu Google Drive o a data/ y volver a ejecutar.")

def montar_drive():
    if not EN_COLAB or not DRIVE_DATA_DIR:
        return None
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        d = Path("/content/drive/MyDrive") / DRIVE_DATA_DIR
        print("Google Drive montado; carpeta de datos:", d, "(existe)" if d.exists() else "(NO existe)")
        return d if d.exists() else None
    except Exception as e:
        print("No se pudo montar Google Drive:", e); return None

def asegurar_paquete(clave, nombre_zip, zip_local, destino):
    # Garantiza que exista <clave> (train.csv o test.csv) bajo DATA_DIR, probando las fuentes en orden
    # 1) ya está descomprimido en data/
    if hay_csv(DATA_DIR, clave):
        print(f"{clave}: encontrado en {buscar_csv(DATA_DIR, clave).parent}"); return
    # 2) zip en data/
    if zip_local.exists():
        print(f"{clave}: descomprimiendo {zip_local}"); descomprimir(zip_local, destino)
        if hay_csv(destino, clave): return
        print(f"  {zip_local} no contenía {clave}; contenido:\n{listar(destino)}")
    # 3) Google Drive propio
    drive_dir = montar_drive()
    if drive_dir is not None:
        if hay_csv(drive_dir, clave):
            origen = buscar_csv(drive_dir, clave).parent
            print(f"{clave}: copiando carpeta {origen} desde Drive (puede tardar unos minutos)...")
            shutil.copytree(origen, destino, dirs_exist_ok=True); return
        zips = [p for p in drive_dir.rglob(nombre_zip) if _valido(p)]
        if zips:
            print(f"{clave}: copiando {zips[0]} desde Drive..."); shutil.copy(zips[0], zip_local)
            descomprimir(zip_local, destino)
            if hay_csv(destino, clave): return
    # 4) descarga pública
    descargar_publico(nombre_zip, zip_local)
    descomprimir(zip_local, destino)
    if not hay_csv(destino, clave):
        raise FileNotFoundError(f"Se descomprimió {nombre_zip} pero no contiene {clave}. Contenido:\n{listar(destino)}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
asegurar_paquete("train.csv", "caso2_train.zip", TRAIN_ZIP, DATA_DIR / "train")
asegurar_paquete("test.csv",  "caso2_test.zip",  TEST_ZIP,  DATA_DIR / "test")

train_csv  = buscar_csv(DATA_DIR, "train.csv")
test_csv   = buscar_csv(DATA_DIR, "test.csv")
clases_csv = buscar_csv(train_csv.parent, "clases.csv")
ejemplo_csv = buscar_csv(test_csv.parent, "ejemplo_predicciones.csv") if hay_csv(test_csv.parent, "ejemplo_predicciones.csv") else None
TRAIN_DIR, TEST_DIR = train_csv.parent, test_csv.parent     # carpeta que contiene cada paquete

# Los CSV de la cátedra traen BOM UTF-8: encoding="utf-8-sig" evita que la primera columna se lea como "\ufeffimage_id"
train_df_raw = pd.read_csv(train_csv, encoding="utf-8-sig")
clases_df    = pd.read_csv(clases_csv, encoding="utf-8-sig")
test_df_raw  = pd.read_csv(test_csv, encoding="utf-8-sig")
img_train = indexar_imagenes(TRAIN_DIR)
img_test  = indexar_imagenes(TEST_DIR)

print("train.csv :", train_csv, train_df_raw.shape, list(train_df_raw.columns))
print("clases.csv:", clases_csv, clases_df.shape, list(clases_df.columns))
print("test.csv  :", test_csv, test_df_raw.shape, list(test_df_raw.columns))
print("imágenes  : train", len(img_train), "| test", len(img_test))
display(train_df_raw.head(3)); display(clases_df.head(3)); display(test_df_raw.head(3))


In [ ]:
# Detección de columnas (identificador y raza) sin depender del nombre exacto
def detectar_col(df, candidatos, descripcion):
    for c in df.columns:
        if c.lower() in candidatos:
            return c
    for c in df.columns:
        if any(k in c.lower() for k in candidatos):
            return c
    raise KeyError(f"No se pudo detectar la columna de {descripcion} en {list(df.columns)}")

COL_ID   = detectar_col(train_df_raw, ["image_id", "id", "imagen", "image", "file", "filename"], "identificador")
COL_RAZA = detectar_col(train_df_raw, ["raza", "breed", "label", "clase", "class"], "raza")
COL_ID_TEST = detectar_col(test_df_raw, ["image_id", "id", "imagen", "image", "file", "filename"], "identificador (test)")
COL_CLASE = detectar_col(clases_df, ["raza", "breed", "clase", "class", "label", "nombre"], "raza (clases.csv)")

CLASES = [str(c) for c in clases_df[COL_CLASE].tolist()]   # orden fijo = el de clases.csv
CLASE_A_IDX = {c: i for i, c in enumerate(CLASES)}
NUM_CLASES = len(CLASES)

def resolver_rutas(df, col_id, indice):
    rutas = []
    for v in df[col_id].astype(str):
        stem = Path(v).stem
        if stem not in indice:
            raise FileNotFoundError(f"No se encontró la imagen de '{v}'")
        rutas.append(str(indice[stem]))
    return rutas

train_df = train_df_raw.copy()
train_df["ruta"] = resolver_rutas(train_df, COL_ID, img_train)
train_df["raza"] = train_df[COL_RAZA].astype(str)
desconocidas = set(train_df["raza"]) - set(CLASES)
assert not desconocidas, f"Razas en train.csv que no están en clases.csv: {desconocidas}"
train_df["y"] = train_df["raza"].map(CLASE_A_IDX)

test_df = test_df_raw.copy()
test_df["ruta"] = resolver_rutas(test_df, COL_ID_TEST, img_test)

# Especie: si clases.csv la trae se usa; si no, convención de Oxford-IIIT Pet (gatos con mayúscula inicial)
col_esp = next((c for c in clases_df.columns if "espec" in c.lower() or "species" in c.lower()), None)
if col_esp:
    ESPECIE = dict(zip(CLASES, clases_df[col_esp].astype(str)))
    print("Especie tomada de clases.csv, columna", col_esp)
else:
    ESPECIE = {c: ("gato" if c[0].isupper() else "perro") for c in CLASES}
    print("Especie derivada de la convención de nombres del dataset (mayúscula inicial = gato)")
train_df["especie"] = train_df["raza"].map(ESPECIE)

print(f"{NUM_CLASES} razas | {len(train_df)} imágenes de train | {len(test_df)} de test")
print("Columna id:", COL_ID, "| columna raza:", COL_RAZA)


## 4. Exploración de los datos

Se mira lo que condiciona decisiones de preparación y modelado: cuántas imágenes hay por raza y especie,
qué tamaño y proporción tienen las fotos, cómo se ven las razas que la consigna marca como parecidas, y si
hay imágenes repetidas (dentro de train y entre train y test, esto último solo para detectar fuga, sin usar
etiquetas).


In [ ]:
# Figura 1: imágenes por raza, coloreadas por especie
conteo = train_df.groupby(["raza", "especie"]).size().reset_index(name="n").sort_values("n", ascending=False)
paleta = {"gato": "#4C72B0", "perro": "#DD8452"} if set(conteo["especie"]) <= {"gato", "perro"} else None
fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=conteo, y="raza", x="n", hue="especie", dodge=False, ax=ax, palette=paleta)
ax.set_title("Figura 1. Cantidad de imágenes por raza en train (color = especie)")
ax.set_xlabel("imágenes"); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig01_imagenes_por_raza.png", dpi=150); plt.show()

resumen_clases = conteo.set_index("raza")["n"]
print(f"Imágenes por raza: mín {resumen_clases.min()} ({resumen_clases.idxmin()}), "
      f"máx {resumen_clases.max()} ({resumen_clases.idxmax()}), mediana {resumen_clases.median():.0f}")
print("Por especie:", train_df["especie"].value_counts().to_dict(),
      "| razas por especie:", pd.Series(ESPECIE).value_counts().to_dict())


In [ ]:
# Figura 2: tamaño y proporción de las fotos (muestra aleatoria para no abrir todas)
rng = np.random.RandomState(SEED)
muestra = train_df.sample(min(600, len(train_df)), random_state=SEED)
dims = []
for r in muestra.itertuples():
    with Image.open(r.ruta) as im:
        dims.append((im.width, im.height, im.mode))
dims = pd.DataFrame(dims, columns=["ancho", "alto", "modo"])
dims["aspecto"] = dims["ancho"] / dims["alto"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(dims["ancho"], dims["alto"], s=8, alpha=0.5)
axes[0].set_xlabel("ancho (px)"); axes[0].set_ylabel("alto (px)"); axes[0].set_title("Ancho vs alto")
sns.histplot(dims["aspecto"], bins=30, ax=axes[1]); axes[1].axvline(1, color="k", ls="--", lw=1)
axes[1].set_xlabel("ancho / alto"); axes[1].set_title("Relación de aspecto")
fig.suptitle("Figura 2. Tamaño y proporción de las fotografías (muestra de train)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig02_tamanos.png", dpi=150); plt.show()

print(dims[["ancho", "alto", "aspecto"]].describe().round(2))
print("Modos de color:", dims["modo"].value_counts().to_dict())
print(f"Lado menor: mediana {np.median(np.minimum(dims['ancho'], dims['alto'])):.0f} px; "
      f"{(np.minimum(dims['ancho'], dims['alto']) < IMG_SIZE).mean()*100:.1f} % de las fotos tiene el lado menor por debajo de {IMG_SIZE} px")


In [ ]:
def mostrar_grilla(rutas, titulos, ncol, titulo_fig, archivo, tam=2.4):
    n = len(rutas); nfil = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nfil, ncol, figsize=(ncol * tam, nfil * tam + 0.6))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, ruta, t in zip(axes, rutas, titulos):
        with Image.open(ruta) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
        ax.set_title(t, fontsize=8)
    fig.suptitle(titulo_fig, fontsize=11)
    plt.tight_layout(); plt.savefig(FIG_DIR / archivo, dpi=130); plt.show()

# Figura 3: una foto por raza (variación de pose, fondo e iluminación)
una_por_raza = train_df.sample(frac=1, random_state=SEED).drop_duplicates("raza").sort_values("raza")
mostrar_grilla(una_por_raza["ruta"].tolist(), una_por_raza["raza"].tolist(), 7,
               "Figura 3. Una fotografía por raza (train)", "fig03_una_por_raza.png", tam=2.0)

# Figura 4: pares de razas visualmente parecidas (según la consigna y la publicación original)
PARES_PARECIDOS = [("american_pit_bull_terrier", "staffordshire_bull_terrier"),
                   ("British_Shorthair", "Russian_Blue"),
                   ("Birman", "Ragdoll"),
                   ("beagle", "basset_hound"),
                   ("boxer", "american_bulldog")]
rutas, titulos = [], []
for a, b in PARES_PARECIDOS:
    if a in CLASE_A_IDX and b in CLASE_A_IDX:
        for raza in (a, b):
            sub = train_df[train_df["raza"] == raza].sample(2, random_state=SEED)
            rutas += sub["ruta"].tolist(); titulos += [raza] * 2
mostrar_grilla(rutas, titulos, 4, "Figura 4. Pares de razas visualmente parecidas (2 fotos de cada una por fila)",
               "fig04_razas_parecidas.png", tam=2.2)


In [ ]:
# Duplicados exactos (hash del archivo) y casi duplicados (hash perceptual promedio de 8x8)
def hash_archivo(ruta):
    return hashlib.md5(Path(ruta).read_bytes()).hexdigest()

# Hash perceptual promedio (aHash): Krawetz (2011), "Looks Like It" — reducir a 8x8 en gris y comparar cada píxel con la media
def ahash(ruta, lado=8):
    with Image.open(ruta) as im:
        g = ImageOps.exif_transpose(im).convert("L").resize((lado, lado), Image.BILINEAR)
    a = np.asarray(g, dtype=np.float32)
    return int("".join("1" if v else "0" for v in (a > a.mean()).flatten()), 2)

def hamming(a, b):
    return bin(a ^ b).count("1")

train_df["md5"] = train_df["ruta"].map(hash_archivo)
train_df["ahash"] = train_df["ruta"].map(ahash)
test_df["md5"] = test_df["ruta"].map(hash_archivo)
test_df["ahash"] = test_df["ruta"].map(ahash)

dup_exactos_train = train_df[train_df.duplicated("md5", keep=False)].sort_values("md5")
dup_exactos_cruzados = test_df[test_df["md5"].isin(set(train_df["md5"]))]

# casi duplicados dentro de train: misma ahash o distancia de Hamming <= 3
UMBRAL_HAMMING = 3
HASH_DEGENERADOS = {0, 2**64 - 1}       # imágenes planas: el hash no dice nada, no se agrupan
grupos = {}
for i, h in zip(train_df.index, train_df["ahash"]):
    if h not in HASH_DEGENERADOS:
        grupos.setdefault(h, []).append(i)
casi_dup_pares = [(ids[0], ids[1]) for ids in grupos.values() if len(ids) > 1]
hashes_train_np = np.array(list(grupos), dtype=np.uint64)
def distancia_minima(ht):
    x = (hashes_train_np ^ np.uint64(ht)).view(np.uint8).reshape(-1, 8)
    return int(np.unpackbits(x, axis=1).sum(1).min())
cruzados_cercanos = sum(1 for ht in test_df["ahash"] if distancia_minima(ht) <= UMBRAL_HAMMING)

print(f"Duplicados exactos dentro de train: {len(dup_exactos_train)} imágenes en "
      f"{dup_exactos_train['md5'].nunique()} grupos")
print(f"Casi duplicados dentro de train (mismo ahash 8x8): {len(casi_dup_pares)} pares")
print(f"Imágenes de test idénticas (md5) a alguna de train: {len(dup_exactos_cruzados)}")
print(f"Imágenes de test a distancia de Hamming <= {UMBRAL_HAMMING} de alguna de train: {cruzados_cercanos}")

if casi_dup_pares:
    sel = casi_dup_pares[:4]
    rutas = [train_df.loc[i, "ruta"] for par in sel for i in par]
    titulos = [f"{train_df.loc[i, 'raza']}\n{Path(train_df.loc[i, 'ruta']).name}" for par in sel for i in par]
    mostrar_grilla(rutas, titulos, 4, "Figura 5. Pares casi duplicados detectados en train (mismo ahash)",
                   "fig05_casi_duplicados.png", tam=2.3)
    etiquetas_distintas = sum(1 for a, b in casi_dup_pares if train_df.loc[a, "raza"] != train_df.loc[b, "raza"])
    print(f"Pares casi duplicados con etiqueta distinta (posible ambigüedad): {etiquetas_distintas}")

# Para que un casi duplicado no quede un lado en train y otro en validación, las fotos con el mismo ahash
# comparten un "grupo"; las demás forman su propio grupo
repetidos = {h for h, ids in grupos.items() if len(ids) > 1}
train_df["grupo_dup"] = [f"h{h}" if h in repetidos else f"i{i}" for i, h in zip(train_df.index, train_df["ahash"])]
eda_resumen = {
    "imagenes_train": int(len(train_df)), "imagenes_test": int(len(test_df)), "razas": NUM_CLASES,
    "min_por_raza": int(resumen_clases.min()), "max_por_raza": int(resumen_clases.max()),
    "dup_exactos_train": int(len(dup_exactos_train)), "casi_dup_pares_train": int(len(casi_dup_pares)),
    "test_identicas_a_train": int(len(dup_exactos_cruzados)), "test_cercanas_a_train": int(cruzados_cercanos),
    "lado_menor_mediana_px": float(np.median(np.minimum(dims["ancho"], dims["alto"]))),
}
(OUT_DIR / "eda_resumen.json").write_text(json.dumps(eda_resumen, indent=2))
eda_resumen


## 5. Partición de entrenamiento y validación

Se reserva el 20 % de `train.csv` para validación, estratificado por raza y con semilla fija, de modo que
cada raza tenga en validación la misma proporción que en train. Los pares casi duplicados detectados arriba
se mantienen del mismo lado de la partición para que la validación no esté "contaminada" con fotos
prácticamente iguales a las de entrenamiento. La partición se guarda en `salidas/split_train_val.csv` y es la
misma para todos los experimentos.


In [ ]:
# Estratificación por raza a nivel de grupo (un grupo = fotos con el mismo ahash; casi siempre 1 foto)
grupos_df = (train_df.groupby("grupo_dup").agg(raza=("raza", "first"), n=("ruta", "size")).reset_index())
g_train, g_val = train_test_split(grupos_df["grupo_dup"], test_size=VAL_FRAC, random_state=SEED,
                                  stratify=grupos_df["raza"])
train_df["split"] = np.where(train_df["grupo_dup"].isin(set(g_val)), "val", "train")

if QUICK_TEST:
    train_df = train_df.groupby(["split", "raza"], group_keys=False).sample(frac=0.10, random_state=SEED)
    print("QUICK_TEST activo: se usa un 10 % de las imágenes")

split_cols = [COL_ID, "raza", "split"]
train_df[split_cols].to_csv(OUT_DIR / "split_train_val.csv", index=False)
df_tr = train_df[train_df["split"] == "train"].reset_index(drop=True)
df_va = train_df[train_df["split"] == "val"].reset_index(drop=True)

tabla_split = pd.crosstab(train_df["raza"], train_df["split"])
tabla_split["% val"] = (tabla_split["val"] / tabla_split.sum(axis=1) * 100).round(1)
print(f"Train: {len(df_tr)} | Validación: {len(df_va)} | "
      f"% val por raza: mín {tabla_split['% val'].min()}, máx {tabla_split['% val'].max()}")
display(tabla_split.sort_values("train"))


## 6. Preparación de imágenes

**Resolución.** 224 × 224, la resolución con la que se preentrenaron los pesos de ImageNet que se reutilizan;
las fotos del dataset tienen el lado menor bastante mayor (ver Figura 2), así que se reduce y no se amplía.

**Normalización.** Media y desvío por canal de ImageNet, porque son los valores que vieron los pesos
preentrenados. Se aplica igual en train, validación y test; no se aprende nada de nuestros datos.

**Aumentos (solo en entrenamiento).** Recorte aleatorio con escala entre 0,65 y 1 (simula distinto encuadre
y tamaño del animal), espejado horizontal (un gato mirando a la izquierda sigue siendo el mismo gato), y
variación leve de brillo, contraste y saturación (iluminación). Se descartan el espejado vertical, las
rotaciones grandes y los cambios fuertes de tono: los animales aparecen de pie y el color del pelaje es
una señal que distingue razas, así que un aumento agresivo podría cambiar la etiqueta.

**Validación y test.** Redimensionado a 256 y recorte central de 224, sin aumentos. Es la receta estándar y
garantiza que validación y test se procesan de forma idéntica.


In [ ]:
# Normalización de los pesos preentrenados (ImageNet), igual en train, validación y test:
# TensorFlow, "Transfer learning and fine-tuning" (usar el preprocesamiento del modelo base); valores de torchvision.
MEDIA_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET   = [0.229, 0.224, 0.225]

# Aumentos que conservan la cara del animal (la región más informativa según Parkhi et al., 2012) y la etiqueta
# (criterio de seguridad de etiqueta de Shorten y Khoshgoftaar, 2019). Solo se aplican al cargador de entrenamiento,
# como en TensorFlow, "Transfer learning and fine-tuning".
tf_train_aug = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(3/4, 4/3)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_IMAGENET, STD_IMAGENET),
])
tf_train_sin_aug = transforms.Compose([
    transforms.Resize(RESIZE_EVAL),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEDIA_IMAGENET, STD_IMAGENET),
])
tf_eval = tf_train_sin_aug

class PetDataset(Dataset):
    def __init__(self, df, transform, con_etiqueta=True):
        self.rutas = df["ruta"].tolist()
        self.y = df["y"].astype(int).tolist() if con_etiqueta else [-1] * len(df)
        self.transform = transform
    def __len__(self):
        return len(self.rutas)
    def __getitem__(self, i):
        with Image.open(self.rutas[i]) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
        return self.transform(im), self.y[i]

def _worker_init(worker_id):
    s = SEED + worker_id
    np.random.seed(s); random.seed(s); torch.manual_seed(s)

def hacer_loader(df, transform, shuffle, con_etiqueta=True):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(PetDataset(df, transform, con_etiqueta), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                      worker_init_fn=_worker_init, generator=g, drop_last=False)

loader_val = hacer_loader(df_va, tf_eval, shuffle=False)

# Figura 6: cómo se ve una misma foto con los aumentos de entrenamiento
def desnormalizar(t):
    t = t.clone()
    for c in range(3): t[c] = t[c] * STD_IMAGENET[c] + MEDIA_IMAGENET[c]
    return t.clamp(0, 1).permute(1, 2, 0).numpy()

fijar_semillas()
ruta_ej = df_tr["ruta"].iloc[0]
with Image.open(ruta_ej) as im:
    im = ImageOps.exif_transpose(im).convert("RGB")
fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
axes[0].imshow(desnormalizar(tf_eval(im))); axes[0].set_title("validación/test")
for ax in axes[1:]:
    ax.imshow(desnormalizar(tf_train_aug(im))); ax.set_title("aumento")
for ax in axes: ax.axis("off")
fig.suptitle(f"Figura 6. Preparación de una foto ({df_tr['raza'].iloc[0]}): recorte central vs aumentos de entrenamiento")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig06_aumentos.png", dpi=130); plt.show()


## 7. Modelos y rutina de entrenamiento

Se comparan dos caminos: una **CNN entrenada desde cero** (contraste) y **transferencia de aprendizaje** con
pesos de ImageNet. Dentro del segundo camino se prueban dos arquitecturas (ResNet50 y ConvNeXt-Tiny), el
mismo modelo con y sin aumentos, y el backbone congelado frente al ajuste completo.

Pesos preentrenados (fuente y versión): `torchvision.models`, `ResNet50_Weights.IMAGENET1K_V2` y
`ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. La versión de torchvision queda registrada en `requirements.txt`.

Rutina común a todos los experimentos: AdamW, tasa de aprendizaje con calentamiento de una época y descenso
coseno, suavizado de etiquetas 0,1, precisión mixta en GPU, y parada temprana por F1 macro en validación.
En el ajuste completo el backbone usa una tasa diez veces menor que la cabeza nueva, para no destruir lo
aprendido en ImageNet (idea tomada del tutorial de TensorFlow sobre fine-tuning). Se guarda el mejor estado
según F1 macro de validación, no el último.


In [ ]:
class CNNDesdeCero(nn.Module):
    '''CNN compacta (~1,5 M parámetros): 5 bloques conv-BN-ReLU-pool y una capa lineal.'''
    def __init__(self, num_clases):
        super().__init__()
        def bloque(ci, co):
            return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                                 nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(bloque(3, 32), bloque(32, 64), bloque(64, 128), bloque(128, 192), bloque(192, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, num_clases))
    def forward(self, x):
        return self.head(self.features(x))

def construir_modelo(nombre, num_clases, congelar_backbone=False):
    '''Devuelve (modelo, params_backbone, params_cabeza, capa_para_gradcam).'''
    if nombre == "cnn_desde_cero":
        m = CNNDesdeCero(num_clases)
        return m, [], list(m.parameters()), m.features[-1]
    if nombre == "resnet50":
        pesos = models.ResNet50_Weights.IMAGENET1K_V2
        m = models.resnet50(weights=pesos)
        m.fc = nn.Linear(m.fc.in_features, num_clases)
        cabeza = list(m.fc.parameters()); capa_cam = m.layer4
    elif nombre == "convnext_tiny":
        pesos = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        m = models.convnext_tiny(weights=pesos)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_clases)
        cabeza = list(m.classifier[2].parameters()); capa_cam = m.features[-1]
    else:
        raise ValueError(nombre)
    ids_cabeza = {id(p) for p in cabeza}
    backbone = [p for p in m.parameters() if id(p) not in ids_cabeza]
    if congelar_backbone:
        for p in backbone: p.requires_grad_(False)
        backbone = []
    return m, backbone, cabeza, capa_cam

def predecir(modelo, loader, tta=False):
    '''Probabilidades softmax (promedio con espejado si tta=True), predicciones y etiquetas.'''
    modelo.eval(); probs, ys = [], []
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            p = F.softmax(modelo(x).float(), dim=1)
            if tta:
                p = 0.5 * (p + F.softmax(modelo(torch.flip(x, dims=[3])).float(), dim=1))
            probs.append(p.cpu()); ys.append(y)
    probs = torch.cat(probs).numpy(); ys = torch.cat(ys).numpy()
    return probs, probs.argmax(1), ys

def metricas(y_true, y_pred):
    return {"exactitud": accuracy_score(y_true, y_pred), "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)}

def entrenar(nombre_exp, arquitectura, loader_tr, epochs, lr_backbone, lr_cabeza, weight_decay,
             congelar=False, descripcion_prep="", paciencia=PACIENCIA_EARLY_STOP):
    fijar_semillas()
    modelo, p_backbone, p_cabeza, _ = construir_modelo(arquitectura, NUM_CLASES, congelar_backbone=congelar)
    modelo.to(DEVICE)
    # Tasa de aprendizaje menor en el backbone preentrenado que en la cabeza nueva: adaptación de la regla del tutorial
    # de TensorFlow "Transfer learning and fine-tuning" (fine-tuning con lr 10 veces menor). En lugar de dos etapas
    # secuenciales se usan dos grupos de parámetros y un calentamiento lineal de una época (abajo).
    grupos = [{"params": p_cabeza, "lr": lr_cabeza}]
    if p_backbone:
        grupos.append({"params": p_backbone, "lr": lr_backbone})
    opt = torch.optim.AdamW(grupos, weight_decay=weight_decay)
    pasos_epoca = len(loader_tr); total = epochs * pasos_epoca; calentamiento = pasos_epoca
    def lr_lambda(paso):
        if paso < calentamiento:
            return (paso + 1) / calentamiento
        prog = (paso - calentamiento) / max(1, total - calentamiento)
        return 0.5 * (1 + np.cos(np.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    # AdamW + coseno + suavizado de etiquetas 0,1: receta de ConvNeXt (Liu et al., 2022); el suavizado también mejora
    # la calibración (Müller, Kornblith y Hinton, 2019).
    criterio = nn.CrossEntropyLoss(label_smoothing=0.1)
    try:
        scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    n_params = sum(p.numel() for p in modelo.parameters())
    n_entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    print(f"\n=== {nombre_exp}: {arquitectura} | {descripcion_prep} | {n_entrenables/1e6:.2f} M parámetros entrenables de {n_params/1e6:.2f} M ===")

    historia = []; mejor_f1 = -1; mejor_estado = None; sin_mejora = 0; t0 = time.time()
    for ep in range(1, epochs + 1):
        modelo.train(); perdida_acum = 0.0; n = 0
        for x, y in loader_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                perdida = criterio(modelo(x), y)
            scaler.scale(perdida).backward(); scaler.step(opt); scaler.update(); sched.step()
            perdida_acum += perdida.item() * len(y); n += len(y)
        _, pred_va, y_va = predecir(modelo, loader_val)
        m = metricas(y_va, pred_va)
        historia.append({"epoca": ep, "perdida_train": perdida_acum / n, **m})
        marca = ""
        if m["f1_macro"] > mejor_f1:
            mejor_f1 = m["f1_macro"]; mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
            sin_mejora = 0; marca = " *"
        else:
            sin_mejora += 1
        print(f"  época {ep:2d} | pérdida {perdida_acum/n:.3f} | val exactitud {m['exactitud']:.4f} | val F1 macro {m['f1_macro']:.4f}{marca}")
        if sin_mejora >= paciencia:
            print(f"  parada temprana: {paciencia} épocas sin mejorar F1 macro"); break
    minutos = (time.time() - t0) / 60
    modelo.load_state_dict(mejor_estado)
    probs_va, pred_va, y_va = predecir(modelo, loader_val)
    m = metricas(y_va, pred_va)
    fila = {"experimento": nombre_exp, "arquitectura": arquitectura, "preparacion": descripcion_prep,
            "congelado": congelar, "epocas_corridas": len(historia), "mejor_epoca": int(np.argmax([h["f1_macro"] for h in historia]) + 1),
            "params_entrenables_M": round(n_entrenables / 1e6, 2), "minutos": round(minutos, 1),
            "val_exactitud": round(m["exactitud"], 4), "val_f1_macro": round(m["f1_macro"], 4)}
    print(f"  -> mejor F1 macro {m['f1_macro']:.4f} (exactitud {m['exactitud']:.4f}) en {minutos:.1f} min")
    return modelo, fila, pd.DataFrame(historia)


## 8. Experimentos

Todos se evalúan con las **mismas imágenes de validación** (`loader_val`) y las **mismas métricas**
(exactitud y F1 macro). La tabla resultante se guarda en `salidas/experimentos.csv` y es la que va al informe.

| Exp. | Arquitectura | Preparación | Qué contrasta |
| --- | --- | --- | --- |
| E1 | CNN desde cero | con aumentos | ¿Cuánto aporta el preentrenamiento? |
| E2 | ResNet50 (ImageNet) ajuste completo | con aumentos | Segunda arquitectura preentrenada |
| E3 | ConvNeXt-Tiny (ImageNet) ajuste completo | con aumentos | Candidato a modelo final |
| E4 | ConvNeXt-Tiny ajuste completo | **sin** aumentos | ¿Cuánto aportan los aumentos? |
| E5 | ConvNeXt-Tiny backbone congelado | con aumentos | ¿Alcanza con entrenar solo la capa final? |


In [ ]:
registro = []; historias = {}; modelos = {}
ep = {k: (1 if QUICK_TEST else v) for k, v in EPOCHS.items()}
loader_tr_aug = hacer_loader(df_tr, tf_train_aug, shuffle=True)
loader_tr_sin = hacer_loader(df_tr, tf_train_sin_aug, shuffle=True)

if RUN_E1_CNN_DESDE_CERO:
    m, fila, h = entrenar("E1", "cnn_desde_cero", loader_tr_aug, ep["E1"], lr_backbone=0, lr_cabeza=1e-3,
                          weight_decay=1e-4, descripcion_prep="224px, norm. ImageNet, con aumentos", paciencia=6)
    registro.append(fila); historias["E1"] = h; modelos["E1"] = m
if RUN_E2_RESNET50_FT:
    m, fila, h = entrenar("E2", "resnet50", loader_tr_aug, ep["E2"], lr_backbone=1e-4, lr_cabeza=1e-3,
                          weight_decay=1e-4, descripcion_prep="224px, norm. ImageNet, con aumentos")
    registro.append(fila); historias["E2"] = h; modelos["E2"] = m
if RUN_E3_CONVNEXT_FT:
    m, fila, h = entrenar("E3", "convnext_tiny", loader_tr_aug, ep["E3"], lr_backbone=1e-4, lr_cabeza=1e-3,
                          weight_decay=0.05, descripcion_prep="224px, norm. ImageNet, con aumentos")
    registro.append(fila); historias["E3"] = h; modelos["E3"] = m
if RUN_E4_CONVNEXT_SIN_AUG:
    m, fila, h = entrenar("E4", "convnext_tiny", loader_tr_sin, ep["E4"], lr_backbone=1e-4, lr_cabeza=1e-3,
                          weight_decay=0.05, descripcion_prep="224px, norm. ImageNet, SIN aumentos")
    registro.append(fila); historias["E4"] = h; modelos["E4"] = m
if RUN_E5_CONVNEXT_CONGELADO:
    m, fila, h = entrenar("E5", "convnext_tiny", loader_tr_aug, ep["E5"], lr_backbone=0, lr_cabeza=1e-3,
                          weight_decay=0.05, congelar=True, descripcion_prep="224px, norm. ImageNet, con aumentos, backbone congelado")
    registro.append(fila); historias["E5"] = h; modelos["E5"] = m

experimentos = pd.DataFrame(registro).sort_values("val_f1_macro", ascending=False)
experimentos.to_csv(OUT_DIR / "experimentos.csv", index=False)
display(experimentos)


In [ ]:
# Figura 7: curvas de F1 macro en validación por época
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k, h in historias.items():
    axes[0].plot(h["epoca"], h["f1_macro"], marker="o", ms=3, label=k)
    axes[1].plot(h["epoca"], h["perdida_train"], marker="o", ms=3, label=k)
axes[0].set_title("F1 macro en validación"); axes[0].set_xlabel("época"); axes[0].legend()
axes[1].set_title("Pérdida de entrenamiento"); axes[1].set_xlabel("época"); axes[1].legend()
fig.suptitle("Figura 7. Evolución por época de cada experimento (misma validación)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig07_curvas.png", dpi=150); plt.show()


## 9. Selección y guardado del modelo final

El modelo final es el de mayor F1 macro en validación. La elección es automática y queda registrada: si la
tabla cambia, cambia el modelo final, pero la regla es la misma. Se guardan los pesos (`modelo_final.pt`) y
una ficha con todo lo necesario para reconstruirlo (`modelo_final_meta.json`).


In [ ]:
EXP_FINAL = experimentos.iloc[0]["experimento"]
modelo_final = modelos[EXP_FINAL]
ARQ_FINAL = experimentos.iloc[0]["arquitectura"]
print(f"Modelo final: {EXP_FINAL} ({ARQ_FINAL}), F1 macro validación = {experimentos.iloc[0]['val_f1_macro']}")

PESOS_PREENTRENADOS = {"resnet50": "torchvision ResNet50_Weights.IMAGENET1K_V2",
                       "convnext_tiny": "torchvision ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
                       "cnn_desde_cero": "ninguno (entrenada desde cero)"}
torch.save(modelo_final.state_dict(), OUT_DIR / "modelo_final.pt")
meta = {"experimento": EXP_FINAL, "arquitectura": ARQ_FINAL, "pesos_preentrenados": PESOS_PREENTRENADOS[ARQ_FINAL],
        "torchvision": torchvision.__version__, "torch": torch.__version__, "img_size": IMG_SIZE, "resize_eval": RESIZE_EVAL,
        "normalizacion": {"media": MEDIA_IMAGENET, "std": STD_IMAGENET}, "clases": CLASES, "semilla": SEED,
        "tta_espejado": USE_TTA, "configuracion": experimentos.iloc[0].to_dict()}
(OUT_DIR / "modelo_final_meta.json").write_text(json.dumps(meta, indent=2, default=str))

# Cómo reconstruirlo en otro script:
#   m, *_ = construir_modelo(meta["arquitectura"], len(meta["clases"]))
#   m.load_state_dict(torch.load("salidas/modelo_final.pt", map_location="cpu")); m.eval()


## 10. Resultados sobre validación: métricas por raza y matriz de confusión

Se reportan exactitud global, F1 macro, precisión y recall por raza y la matriz de confusión del modelo
final sobre las imágenes de validación, con la misma inferencia que se usará en test (espejado horizontal
promediado si `USE_TTA`).


In [ ]:
probs_va, pred_va, y_va = predecir(modelo_final, loader_val, tta=USE_TTA)
m_final = metricas(y_va, pred_va)
sin_tta = metricas(y_va, predecir(modelo_final, loader_val, tta=False)[1])
print(f"Modelo final sobre validación | exactitud {m_final['exactitud']:.4f} | F1 macro {m_final['f1_macro']:.4f}"
      f" | (sin TTA: exactitud {sin_tta['exactitud']:.4f}, F1 macro {sin_tta['f1_macro']:.4f})")

prec, rec, f1, sop = precision_recall_fscore_support(y_va, pred_va, labels=list(range(NUM_CLASES)), zero_division=0)
por_raza = pd.DataFrame({"raza": CLASES, "especie": [ESPECIE[c] for c in CLASES], "n_val": sop,
                         "precision": prec.round(3), "recall": rec.round(3), "f1": f1.round(3)}).sort_values("f1")
por_raza.to_csv(OUT_DIR / "metricas_por_raza.csv", index=False)
print("\nRazas con menor F1:"); display(por_raza.head(8))
print("Razas con mayor F1:"); display(por_raza.tail(5))
print("F1 macro por especie:", por_raza.groupby("especie")["f1"].mean().round(3).to_dict())

# Figura 8: matriz de confusión normalizada por fila (recall en la diagonal)
cm = confusion_matrix(y_va, pred_va, labels=list(range(NUM_CLASES)))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(cm_norm, xticklabels=CLASES, yticklabels=CLASES, cmap="Blues", vmin=0, vmax=1, square=True,
            cbar_kws={"shrink": 0.6}, ax=ax, linewidths=0.2, linecolor="#eee")
# anotar solo celdas fuera de la diagonal con al menos 2 errores, para que sea legible
for i in range(NUM_CLASES):
    for j in range(NUM_CLASES):
        if i != j and cm[i, j] >= 2:
            ax.text(j + 0.5, i + 0.5, str(cm[i, j]), ha="center", va="center", fontsize=7, color="#b00")
ax.set_xlabel("raza predicha"); ax.set_ylabel("raza real")
ax.set_title("Figura 8. Matriz de confusión sobre validación (normalizada por fila; números rojos = errores ≥ 2)")
plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR / "fig08_matriz_confusion.png", dpi=150); plt.show()

# Confusiones más frecuentes
conf = [(CLASES[i], CLASES[j], int(cm[i, j])) for i in range(NUM_CLASES) for j in range(NUM_CLASES) if i != j and cm[i, j] > 0]
confusiones = pd.DataFrame(conf, columns=["real", "predicha", "n"]).sort_values("n", ascending=False)
confusiones["misma_especie"] = [ESPECIE[a] == ESPECIE[b] for a, b in zip(confusiones["real"], confusiones["predicha"])]
confusiones.to_csv(OUT_DIR / "confusiones_top.csv", index=False)
print(f"Errores totales: {int(cm.sum() - np.trace(cm))} | entre especies distintas: {int(confusiones.loc[~confusiones['misma_especie'], 'n'].sum())}")
display(confusiones.head(12))


## 11. Entender los errores: ejemplos y Grad-CAM

Se muestran aciertos con alta confianza, errores con alta confianza (los más costosos para el portal, porque
el criterio de decisión los dejaría pasar) y errores con baja confianza. Luego, Grad-CAM (Selvaraju et al.,
2017) sobre la última etapa convolucional del modelo final muestra qué región de la foto sostiene la
predicción: si el mapa cae sobre la cara y el cuerpo del animal, el modelo se apoya en el animal; si cae
sobre el fondo, el collar o la persona, se apoya en el contexto.


In [ ]:
df_va_res = df_va.copy()
df_va_res["pred"] = [CLASES[i] for i in pred_va]
df_va_res["confianza"] = probs_va.max(1)
df_va_res["acierto"] = df_va_res["pred"] == df_va_res["raza"]

def titulo_fila(r):
    return f"real: {r.raza}\npred: {r.pred} ({r.confianza:.2f})"

aciertos = df_va_res[df_va_res["acierto"]].sort_values("confianza", ascending=False).head(4)
err_conf = df_va_res[~df_va_res["acierto"]].sort_values("confianza", ascending=False).head(4)
err_duda = df_va_res[~df_va_res["acierto"]].sort_values("confianza").head(4)
sel = pd.concat([aciertos, err_conf, err_duda])
mostrar_grilla(sel["ruta"].tolist(), [titulo_fila(r) for r in sel.itertuples()], 4,
               "Figura 9. Fila 1: aciertos confiados · Fila 2: errores confiados · Fila 3: errores con baja confianza",
               "fig09_aciertos_errores.png", tam=2.6)


In [ ]:
class GradCAM:
    '''Implementación mínima de Grad-CAM con hooks (Selvaraju et al., 2017).'''
    def __init__(self, modelo, capa):
        self.modelo = modelo; self.act = None; self.grad = None
        def hook_forward(m, i, o):
            self.act = o
            if o.requires_grad:
                o.register_hook(lambda g: setattr(self, "grad", g))
        self.handle = capa.register_forward_hook(hook_forward)
    def __call__(self, x, clase=None):
        self.modelo.eval(); self.modelo.zero_grad()
        x = x.clone().requires_grad_(True)      # garantiza gradiente aunque el backbone esté congelado
        salida = self.modelo(x)
        if clase is None: clase = salida.argmax(1)
        salida[torch.arange(len(x)), clase].sum().backward()
        pesos = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((pesos * self.act).sum(1))
        cam = F.interpolate(cam[:, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = cam - cam.flatten(1).min(1)[0][:, None, None]
        cam = cam / (cam.flatten(1).max(1)[0][:, None, None] + 1e-8)
        return cam.detach().cpu().numpy(), salida.softmax(1).detach().cpu().numpy()

# última etapa convolucional del modelo final (misma correspondencia que en construir_modelo)
capa_cam_final = {"convnext_tiny": lambda m: m.features[-1], "resnet50": lambda m: m.layer4,
                  "cnn_desde_cero": lambda m: m.features[-1]}[ARQ_FINAL](modelo_final)
gradcam = GradCAM(modelo_final, capa_cam_final)

ejemplos_cam = pd.concat([aciertos.head(3), err_conf.head(3)])
fig, axes = plt.subplots(2, len(ejemplos_cam), figsize=(2.4 * len(ejemplos_cam), 5.2), squeeze=False)
for k, r in enumerate(ejemplos_cam.itertuples()):
    with Image.open(r.ruta) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
    x = tf_eval(im)[None].to(DEVICE)
    with torch.enable_grad():
        cam, _ = gradcam(x.float())
    base = desnormalizar(x[0].cpu())
    axes[0, k].imshow(base); axes[0, k].set_title(titulo_fila(r), fontsize=7)
    axes[1, k].imshow(base); axes[1, k].imshow(cam[0], cmap="jet", alpha=0.45)
    for a in axes[:, k]: a.axis("off")
fig.suptitle("Figura 10. Grad-CAM del modelo final: 3 aciertos (izq.) y 3 errores confiados (der.). Rojo = región que más pesa en la predicción")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig10_gradcam.png", dpi=130); plt.show()
gradcam.handle.remove()   # el modelo vuelve a usarse sin hooks para predecir test


## 12. Criterio de uso: cuándo sugerir y cuándo pedir revisión

La confianza del modelo es la probabilidad softmax de la raza predicha. Como las redes modernas suelen estar
sobreconfiadas (Guo et al., 2017), no se asume que una confianza de 0,9 equivale a 90 % de acierto: se mide
en validación, para cada umbral posible, qué porcentaje de fotos recibiría sugerencia (cobertura) y qué
porcentaje de esas sugerencias sería correcto (precisión de las sugerencias).

**Regla elegida:** el umbral más bajo con el que la precisión de las sugerencias en validación alcanza
`OBJETIVO_PRECISION_SUGERENCIAS` (95 %). Así se maximiza la cobertura sujeta a un tope de error aceptable para
el portal. Con la confianza por debajo del umbral la foto va a revisión, aunque igual se entrega la raza
predicha.


In [ ]:
# Umbral elegido midiendo precisión de las sugerencias sobre validación, no asumiendo que la confianza es una probabilidad
# (sobreconfianza de las redes modernas: Guo et al., 2017). ECE y diagrama de confiabilidad siguen la definición de ese artículo.
conf_va = probs_va.max(1); acierto_va = (pred_va == y_va)
filas = []
for tau in np.round(np.arange(0.30, 1.0001, 0.01), 2):
    sug = conf_va >= tau
    n_sug = int(sug.sum())
    prec_sug = float(acierto_va[sug].mean()) if n_sug else np.nan
    err_sug = int((~acierto_va[sug]).sum())
    err_total = int((~acierto_va).sum())
    filas.append({"umbral": tau, "cobertura": n_sug / len(conf_va), "precision_sugerencias": prec_sug,
                  "errores_entre_sugeridas": err_sug, "errores_derivados_a_revision": err_total - err_sug,
                  "pct_errores_capturados": (err_total - err_sug) / max(err_total, 1)})
umbrales = pd.DataFrame(filas)
umbrales.to_csv(OUT_DIR / "umbrales.csv", index=False)

candidatos = umbrales[umbrales["precision_sugerencias"] >= OBJETIVO_PRECISION_SUGERENCIAS]
TAU = float(candidatos["umbral"].min()) if len(candidatos) else 1.0
fila_tau = umbrales[umbrales["umbral"] == TAU].iloc[0]
print(f"Umbral elegido τ = {TAU:.2f} (objetivo: precisión de sugerencias ≥ {OBJETIVO_PRECISION_SUGERENCIAS:.0%})")
print(f"  Cobertura (fotos con sugerencia): {fila_tau['cobertura']:.1%}")
print(f"  Precisión de las sugerencias     : {fila_tau['precision_sugerencias']:.1%}  "
      f"({int(fila_tau['errores_entre_sugeridas'])} errores entre las sugeridas)")
print(f"  Errores que caen en revisión     : {int(fila_tau['errores_derivados_a_revision'])} de {int((~acierto_va).sum())} "
      f"({fila_tau['pct_errores_capturados']:.1%})")
display(umbrales[umbrales["umbral"].isin([0.5, 0.6, 0.7, 0.8, 0.9, 0.95, TAU])].round(3).sort_values("umbral"))

# Figura 11: cobertura y precisión según el umbral + diagrama de confiabilidad (calibración)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(umbrales["umbral"], umbrales["cobertura"], label="cobertura (recibe sugerencia)")
axes[0].plot(umbrales["umbral"], umbrales["precision_sugerencias"], label="precisión de las sugerencias")
axes[0].axvline(TAU, color="k", ls="--", lw=1, label=f"τ = {TAU:.2f}")
axes[0].axhline(OBJETIVO_PRECISION_SUGERENCIAS, color="gray", ls=":", lw=1)
axes[0].set_xlabel("umbral de confianza"); axes[0].set_ylim(0, 1.02); axes[0].legend(loc="lower left")
axes[0].set_title("Cobertura y precisión según el umbral (validación)")
bins = np.linspace(0, 1, 11); idx = np.clip(np.digitize(conf_va, bins) - 1, 0, 9)
calib = pd.DataFrame({"bin": idx, "conf": conf_va, "acierto": acierto_va}).groupby("bin").agg(conf_media=("conf", "mean"), exactitud=("acierto", "mean"), n=("conf", "size"))
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="calibración perfecta")
axes[1].scatter(calib["conf_media"], calib["exactitud"], s=calib["n"] * 2 + 10, alpha=0.8, label="tramos (tamaño = n)")
axes[1].set_xlabel("confianza media del tramo"); axes[1].set_ylabel("exactitud observada"); axes[1].legend()
axes[1].set_title("Diagrama de confiabilidad")
fig.suptitle("Figura 11. Criterio sugerir/revisar y calibración del modelo final")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig11_umbral_calibracion.png", dpi=150); plt.show()

ece = float((calib["n"] / calib["n"].sum() * (calib["conf_media"] - calib["exactitud"]).abs()).sum())
print(f"Error de calibración esperado (ECE, 10 tramos): {ece:.3f}")

# Razas más frecuentes entre los errores que igual recibirían sugerencia
err_sugeridos = df_va_res[(df_va_res["confianza"] >= TAU) & (~df_va_res["acierto"])]
print("Errores que pasarían como sugerencia, por par (real -> predicha):")
display(err_sugeridos.groupby(["raza", "pred"]).size().sort_values(ascending=False).head(10).rename("n").reset_index())

criterio = {"regla": "sugerir si max(softmax) >= umbral; en caso contrario revisar",
            "umbral": TAU, "objetivo_precision_sugerencias": OBJETIVO_PRECISION_SUGERENCIAS, "tta": USE_TTA,
            "validacion": {"cobertura": float(fila_tau["cobertura"]), "precision_sugerencias": float(fila_tau["precision_sugerencias"]),
                           "errores_entre_sugeridas": int(fila_tau["errores_entre_sugeridas"]),
                           "errores_derivados_a_revision": int(fila_tau["errores_derivados_a_revision"]),
                           "n_val": int(len(conf_va)), "ece": ece}}
(OUT_DIR / "criterio.json").write_text(json.dumps(criterio, indent=2))


## 13. Predicciones sobre test

Única vez que se usa el conjunto de test: se lee `test.csv`, se aplica exactamente la misma preparación que
en validación, se predice con el modelo final y se aplica el umbral fijado arriba. Se escribe
`salidas/predicciones_test.csv` y se valida su formato contra `ejemplo_predicciones.csv`.


In [ ]:
loader_test = hacer_loader(test_df, tf_eval, shuffle=False, con_etiqueta=False)
probs_te, pred_te, _ = predecir(modelo_final, loader_test, tta=USE_TTA)
conf_te = probs_te.max(1)

predicciones = pd.DataFrame({
    "image_id": test_df[COL_ID_TEST].astype(str).values,
    "raza_predicha": [CLASES[i] for i in pred_te],
    "decision": np.where(conf_te >= TAU, "sugerir", "revisar"),
})
predicciones.to_csv(OUT_DIR / "predicciones_test.csv", index=False)
predicciones.to_csv("predicciones_test.csv", index=False)   # copia en la raíz del proyecto para la entrega

# Validaciones de formato
assert list(predicciones.columns) == ["image_id", "raza_predicha", "decision"]
assert len(predicciones) == len(test_df_raw), "debe haber una fila por imagen de test.csv"
assert predicciones["image_id"].is_unique
assert set(predicciones["raza_predicha"]) <= set(CLASES), "raza_predicha debe estar escrita como en clases.csv"
assert set(predicciones["decision"]) <= {"sugerir", "revisar"}
if ejemplo_csv is not None:
    ejemplo = pd.read_csv(ejemplo_csv, encoding="utf-8-sig")
    assert list(ejemplo.columns) == list(predicciones.columns), f"columnas del ejemplo: {list(ejemplo.columns)}"
    print("Formato verificado contra", ejemplo_csv.name)
print(f"predicciones_test.csv: {len(predicciones)} filas | decisión: {predicciones['decision'].value_counts(normalize=True).round(3).to_dict()}")
print(f"Confianza media en test: {conf_te.mean():.3f} (validación: {conf_va.mean():.3f})")
display(predicciones.head())
# Distribución de razas predichas en test (control de cordura: no debería concentrarse en pocas razas)
display(predicciones["raza_predicha"].value_counts().head(10).rename("n").reset_index())


## 14. Archivos generados y prototipo opcional

Resumen de todo lo que quedó en `salidas/`. El prototipo con Gradio es opcional (`RUN_GRADIO = True`): permite
subir una foto y ver la raza sugerida, la confianza y la decisión con el mismo criterio del portal.


In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT_DIR)!s:45} {p.stat().st_size/1024:8.1f} KB")

if RUN_GRADIO:
    _asegurar("gradio")
    import gradio as gr
    def demo_predecir(img):
        x = tf_eval(img.convert("RGB"))[None].to(DEVICE)
        modelo_final.eval()
        with torch.no_grad():
            p = F.softmax(modelo_final(x).float(), 1)
            if USE_TTA: p = 0.5 * (p + F.softmax(modelo_final(torch.flip(x, dims=[3])).float(), 1))
        p = p[0].cpu().numpy(); top = p.argsort()[::-1][:3]
        decision = "sugerir" if p[top[0]] >= TAU else "revisar (pedir revisión humana)"
        return {CLASES[i]: float(p[i]) for i in top}, decision
    gr.Interface(demo_predecir, gr.Image(type="pil"), [gr.Label(num_top_classes=3), gr.Textbox(label="decisión")],
                 title="Sugerencia de raza para el portal de adopción").launch(share=False)
